# Proyecto · Clase 3 — Aprender bajando la montaña (descenso por gradiente)

El perceptrón de la Clase 2 tiene dos problemas:
1. Solo dice **0 o 1**. No dice *qué tan seguro* está.
2. Su regla solo sabe si acertó o no, no **cuánto** se equivocó. Con 3 vs 5 nunca deja de saltar.

Hoy lo arreglamos con las dos ideas que usan **todas** las redes modernas:
- una salida suave, la **sigmoide** → una probabilidad entre 0 y 1;
- un **error que se puede medir** y una forma de bajarlo poco a poco: el **descenso por gradiente**.

In [ ]:
import os, urllib.request
if not os.path.exists("digitos.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/datahuba/curso-redes-neuronales/main/proyecto/digitos.py", "digitos.py")

import numpy as np
import matplotlib.pyplot as plt
from digitos import cargar_mnist, mostrar, ok

## 1. La sigmoide: de «sí/no» a «qué tan seguro»
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
Números muy negativos → casi 0. Muy positivos → casi 1. Cero → 0.5 («no sé»).

✏️ **Ejercicio 1.** Escríbela con `np.exp`.

In [ ]:
def sigmoide(z):
    return ...

z = np.linspace(-8, 8, 200)
plt.plot(z, sigmoide(z), label="sigmoide")
plt.plot(z, (z > 0).astype(int), "--", label="escalón (perceptrón)")
plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
ok(abs(sigmoide(0) - 0.5) < 1e-9 and sigmoide(10) > 0.99 and sigmoide(-10) < 0.01, "la sigmoide va de 0 a 1")

## 2. Descenso por gradiente con un solo número
Imagina que estás en una montaña con niebla y quieres llegar al fondo del valle. Solo puedes sentir **la
pendiente bajo tus pies**. Estrategia: da un paso **cuesta abajo**, vuelve a sentir, repite.

Probémoslo con un error inventado: $E(w) = (w - 3)^2$. Su fondo está en $w = 3$.
Su pendiente (derivada) es $E'(w) = 2(w - 3)$.

$$w \leftarrow w - \eta \cdot \text{pendiente}$$

✏️ **Ejercicio 2.** Completa el paso.

In [ ]:
def bajar(eta, pasos=20, w=-4.0):
    camino = [w]
    for _ in range(pasos):
        pendiente = 2 * (w - 3)
        w = ...
        camino.append(w)
    return np.array(camino)

ws = np.linspace(-5, 11, 200)
for eta in [0.1, 0.5, 1.05]:     # 🔁 prueba otros valores
    c = bajar(eta)
    plt.figure(figsize=(5, 3))
    plt.plot(ws, (ws - 3) ** 2, color="gray")
    plt.plot(c, (c - 3) ** 2, "o-", color="tab:orange")
    plt.ylim(0, 70); plt.title(f"eta = {eta}  →  termina en w = {c[-1]:.2f}"); plt.show()

In [ ]:
ok(abs(bajar(0.1)[-1] - 3) < 0.2, "con eta = 0.1 llega cerca de w = 3")

**Para comentar:** ¿qué pasa con `eta` muy chico? ¿Y con `eta = 1.05`? Elegir la tasa de aprendizaje es una de
las decisiones más importantes al entrenar una red.

## 3. Ahora con 784 pesos: ¿3 o 5?
Mismo truco, pero en una montaña de 784 dimensiones. Para la neurona sigmoide, la pendiente de cada peso
resulta ser muy simple (la deducción está en la guía, no hace falta memorizarla):

$$\text{pendiente de } w = \frac{1}{n} X^\top (p - y) \qquad \text{pendiente de } b = \text{promedio}(p - y)$$

donde `p` es lo que dijo la neurona (probabilidad) e `y` lo correcto. **Error × entrada**, igual que Rosenblatt,
pero ahora el error es *cuánto* se equivocó, no solo si se equivocó.

In [ ]:
X_train, y_train, X_test, y_test = cargar_mnist()
A, B = 3, 5
f, ft = (y_train == A) | (y_train == B), (y_test == A) | (y_test == B)
X, y = X_train[f].reshape(-1, 784) / 255, (y_train[f] == B).astype(float)
Xt, yt = X_test[ft].reshape(-1, 784) / 255, (y_test[ft] == B).astype(float)
print(len(X), "imágenes de", A, "y", B)

Para medir el error usamos la **entropía cruzada**: castiga mucho decir «99 % seguro que es 5» cuando era 3.
(Te la damos hecha.)

In [ ]:
def error_promedio(p, y):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

✏️ **Ejercicio 3.** Completa el entrenamiento: predecir, calcular pendientes, dar el paso.

In [ ]:
def entrenar(X, y, eta=0.5, pasos=300):
    w, b = np.zeros(784), 0.0
    historia = []
    for paso in range(pasos):
        p = ...                       # probabilidad de que cada imagen sea B
        grad_w = ...                  # pendiente de los pesos
        grad_b = ...                  # pendiente del sesgo
        w, b = ..., ...               # un paso cuesta abajo
        historia.append(error_promedio(p, y))
    return w, b, historia

w, b, historia = entrenar(X, y)
plt.plot(historia); plt.xlabel("paso"); plt.ylabel("error"); plt.title("La curva de aprendizaje"); plt.show()

p_test = sigmoide(Xt @ w + b)
precision = ((p_test > 0.5) == yt).mean()
print(f"Precisión 3 vs 5: {precision:.1%}")

In [ ]:
ok(historia[-1] < historia[0] and precision > 0.9, "el error baja y la neurona acierta más del 90 %")

## 4. Ahora sabe cuándo duda
Estas son las imágenes donde la neurona está **menos segura** (probabilidad cerca de 0.5):

In [ ]:
dudas = np.argsort(np.abs(p_test - 0.5))[:10]
mostrar(Xt[dudas], titulos=[f"{p_test[i]:.0%} es {B}" for i in dudas])

## Lo que nos llevamos
- **Sigmoide**: la neurona da una probabilidad, no solo sí/no.
- **Error medible** + **descenso por gradiente**: bajar el error paso a paso, con tamaño de paso `eta`.
- La **curva de aprendizaje** (error vs pasos) es el electrocardiograma del entrenamiento: siempre mírala.

**Clase 4:** ¡basta de preguntas de sí/no! **10 neuronas**, una por dígito, y vas a dibujar tus propios números.